<script src="{{site.baseurl}}/assets/js/code-runner-analytics.js"></script>

## Popcorn Hack 1: SFI Search Filter

A record counts as a match when its spec number equals the query **or** its product name contains the query. Only records from an accepted racing category are returned. I run several different queries to show the search keeps working as the query changes.

In [ ]:
# CODE_RUNNER: Popcorn Hack 1 - SFI Search Filter

query = "clutch"

records = [
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    },
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    }
]

valid_categories = ["Drag Racing", "Auto Racing"]


def search(search_query):
    results = []
    for record in records:
        spec_match = search_query == record["spec_number"]
        name_match = search_query.lower() in record["product_name"].lower()
        accepted_category = record["category"] in valid_categories
        # (spec OR name) AND accepted category
        if (spec_match or name_match) and accepted_category:
            results.append(record)
    return results


for test_query in [query, "2.1", "7.7"]:
    matches = search(test_query)
    print('Searching "' + test_query + '":', len(matches), "match(es)")
    for record in matches:
        print("  ", record["spec_number"], "-", record["product_name"])

## Popcorn Hack 2: Check One SFI Part Record

A record is only complete if it has a racing category, a spec number, an effective date, and a product name. The accepted categories are Drag Racing and Auto Racing. The runner prints True or False for each rule.

In [ ]:
# CODE_RUNNER: Popcorn Hack 2 - Check SFI Part Fields

part = {
    "product_name": "Multiple Disc Clutch Assemblies",
    "category": "Drag Racing",
    "spec_number": "1.2",
    "effective_date": "Feb. 9, 2006"
}

valid_categories = ["Drag Racing", "Auto Racing"]

# One True/False check for each rule.
valid_category = part["category"] in valid_categories
has_spec_number = part["spec_number"].strip() != ""
has_effective_date = part["effective_date"].strip() != ""
has_product_name = part["product_name"].strip() != ""

print("Accepted racing category:", valid_category)
print("Spec number present:", has_spec_number)
print("Effective date present:", has_effective_date)
print("Product name present:", has_product_name)

# The record is complete only if EVERY check is True.
record_is_complete = valid_category and has_spec_number and has_effective_date and has_product_name
print("Record is complete:", record_is_complete)

## Popcorn Hack 3: Backend Create Rule

The backend only creates a new record if its spec number is not already stored **and** every required field is valid. I try it with a duplicate record, a valid record, and an invalid record.

In [ ]:
# CODE_RUNNER: Popcorn Hack 3 - Backend Create Rule

valid_categories = ["Drag Racing", "Auto Racing"]
existing_spec_numbers = ["2.1", "1.1"]


def can_create(record):
    duplicate_spec = record["spec_number"] in existing_spec_numbers
    fields_valid = (
        record["category"] in valid_categories
        and record["spec_number"].strip() != ""
        and record["effective_date"].strip() != ""
        and record["product_name"].strip() != ""
    )
    print("  duplicate_spec:", duplicate_spec, "| fields_valid:", fields_valid)
    # Only create when the spec is NOT a duplicate AND the fields are valid.
    return not duplicate_spec and fields_valid


duplicate_part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_part = {
    "product_name": "Multiple Disc Clutch Assemblies",
    "category": "Drag Racing",
    "spec_number": "1.2",
    "effective_date": "Feb. 9, 2006"
}

invalid_part = {
    "product_name": "Street Flywheel",
    "category": "Street Car",
    "spec_number": "3.1",
    "effective_date": "Jan. 1, 2026"
}

tests = [("duplicate", duplicate_part), ("valid", valid_part), ("invalid", invalid_part)]

for label, record in tests:
    print("Trying (" + label + "):", record["spec_number"], "-", record["product_name"])
    if can_create(record):
        print("  -> record CREATED")
    else:
        print("  -> record REJECTED")

## Popcorn Hack 4 / Homework: SFI Backend Validator

A reusable validator for new records. A record passes when it has an accepted racing category, a spec number that is not already stored, an effective date, and a product name. Any record that fails prints the reasons why.

In [ ]:
# CODE_RUNNER: Popcorn Hack 4 / Homework - SFI Backend Validator

existing_spec_numbers = ["2.1", "1.1"]
valid_categories = ["Drag Racing", "Auto Racing"]

test_records = [
    {
        "product_name": "Replacement Flywheels",
        "category": "Street Car",
        "spec_number": "3.1",
        "effective_date": "Jan. 1, 2026"
    },
    {
        "product_name": "Duplicate Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1",
        "effective_date": "Jan. 1, 2026"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2",
        "effective_date": "Feb. 9, 2006"
    }
]


def validate_record(record):
    valid_category = record["category"] in valid_categories
    has_spec_number = record["spec_number"].strip() != ""
    duplicate_spec = record["spec_number"] in existing_spec_numbers
    has_effective_date = record["effective_date"].strip() != ""
    has_product_name = record["product_name"].strip() != ""

    is_valid = (
        valid_category
        and has_spec_number
        and not duplicate_spec
        and has_effective_date
        and has_product_name
    )

    reasons = []
    if not valid_category:
        reasons.append("'" + record["category"] + "' is not an accepted category")
    if not has_spec_number:
        reasons.append("no spec number")
    if duplicate_spec:
        reasons.append("spec " + record["spec_number"] + " is already stored")
    if not has_effective_date:
        reasons.append("no effective date")
    if not has_product_name:
        reasons.append("no product name")

    return is_valid, reasons


for record in test_records:
    is_valid, reasons = validate_record(record)
    if is_valid:
        existing_spec_numbers.append(record["spec_number"])  # save it
        print("ACCEPTED:", record["spec_number"], "-", record["product_name"])
    else:
        print("REJECTED:", record["spec_number"], "-", record["product_name"], "->", ", ".join(reasons))

print()
print("Specs now stored:", existing_spec_numbers)